# Chunking Strategies
## Stripe API Documentation for Agentic Customer Support RAG
---

## Chunking Strategies Implementation

Implement and compare 5 different chunking strategies:
1. **Fixed-size with overlap** - Simple, predictable
2. **Recursive character splitting** - LangChain's intelligent splitting
3. **Semantic chunking** - LlamaIndex's embedding-based splitting
4. **Sentence-window** - Context preservation technique
5. **Hierarchical parent-child** - Multi-level retrieval structure

---


## Environment Setup

Tokenization involves breaking down text into smaller units, known as tokens, which can be words, subwords, or characters.

**tiktoken** is a fast and efficient tokenization library developed by OpenAI. It provides a robust solution for converting text into tokens and vice versa.

Encoding models in Tiktoken determine the rules for breaking down text into tokens.
- **o200k_base**: Encoding for the newest GPT-4o-Mini model.
- **cl100k_base**: Encoding model for newer OpenAI models such as GPT-4 and GPT-3.5-Turbo.
- **p50k_base**: Encoding for Codex models, these models are used for code applications.
- **r50k_base**: Older encoding for different versions of GPT-3.

Knowing the token count before sending a request to the OpenAI API can help manage costs effectively.

---

**tqdm** is used for progress bars and ipywidgets to enable rich, interactive visualizations inside Jupyter notebooks.

---
**matplotlib** and **seaborn** are used for data visualisation.

---

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import tiktoken
print(tiktoken.encoding_for_model('text-embedding-3-small'))

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter
)

from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, asdict
from pathlib import Path
import json

# .env file
import os
from dotenv import load_dotenv


# StripeDoc hash
import hashlib
from tqdm.auto import tqdm


import pandas as pd
import numpy as np


from llama_index.embeddings.openai import OpenAIEmbedding

from llama_index.core.node_parser import (
    SentenceSplitter,
    SemanticSplitterNodeParser,
    HierarchicalNodeParser
)

from llama_index.core import Document as LlamaDocument


import matplotlib.pyplot as plt

In [ ]:
load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print("API KEY for OPENAI is not set. Please check your .env file.")
else:
    print("OPENAI_API_KEY loaded successfully.")

if not os.environ.get("GROQ_API_KEY"):
    print("API key for Groq is not set. Please check your .env file.")
else:
    print("API key loaded successfully.")

print(os.getenv("OPENAI_API_KEY"))
print(os.environ.get("GROQ_API_KEY"))

## Configuration & Data Models

In [ ]:
@dataclass
class StripeDoc:
    """Data model for scraped Stripe documentation"""
    url: str
    title: str
    content: str
    doc_type: str  # 'api_reference', 'guide', 'tutorial', 'code_example'
    category: str  # 'payments', 'billing', 'connect', etc.
    subcategory: Optional[str] = None
    language: Optional[str] = None  # For code examples: 'python', 'javascript', etc.
    difficulty: Optional[str] = None  # 'beginner', 'intermediate', 'advanced'
    code_blocks: List[str] = None
    scraped_at: str = None
    doc_id: str = None  # Hash-based unique identifier
    
    def __post_init__(self):
        if self.code_blocks is None:
            self.code_blocks = []
        if self.scraped_at is None:
            from datetime import datetime
            self.scraped_at = datetime.now().isoformat()
        if self.doc_id is None:
            self.doc_id = hashlib.md5(self.url.encode()).hexdigest()[:16]

@dataclass
class Chunk:
    """Data model for document chunks"""
    chunk_id: str
    doc_id: str
    content: str
    chunk_index: int
    chunking_strategy: str
    token_count: int
    char_count: int
    metadata: Dict
    parent_chunk_id: Optional[str] = None  # For hierarchical chunking
    child_chunk_ids: List[str] = None  # For hierarchical chunking
    
    def __post_init__(self):
        if self.child_chunk_ids is None:
            self.child_chunk_ids = []

# Configuration
CONFIG = {
    'base_url': 'https://docs.stripe.com',
    'output_dir': Path('./stripe_docs_data'),
    'raw_docs_file': 'raw_documents.json',
    'chunks_dir': 'chunks',
    'rate_limit_delay': 1.0,  # seconds between requests
    'max_docs': 100,  # Limit for demo purposes; set to None for full scrape
    'user_agent': 'RAG-Project',
    'timeout': 30,
    'encoding': 'cl100k_base',  # OpenAI's tiktoken encoding
}

# Create output directories
CONFIG['output_dir'].mkdir(exist_ok=True)
(CONFIG['output_dir'] / CONFIG['chunks_dir']).mkdir(exist_ok=True)

print("✓ Configuration loaded")
print(f"  Output directory: {CONFIG['output_dir']}")
print(f"  Max documents: {CONFIG['max_docs']}")

In [ ]:
# Path to the saved raw documents
input_file = CONFIG['output_dir'] / CONFIG['raw_docs_file']

# Load JSON and reconstruct objects
with open(input_file, 'r', encoding='utf-8') as f:
    raw_docs = json.load(f)

stripe_docs = [StripeDoc(**d) for d in raw_docs]

print(f"✓ Loaded {len(stripe_docs)} documents from {input_file}")


## Chunking Strategies Implementation

We'll implement and compare 5 different chunking strategies:
1. **Fixed-size with overlap** - Simple, predictable
2. **Recursive character splitting** - LangChain's intelligent splitting
3. **Semantic chunking** - LlamaIndex's embedding-based splitting
4. **Sentence-window** - Context preservation technique
5. **Hierarchical parent-child** - Multi-level retrieval structure

In [ ]:
# Initialize tokenizer for accurate token counting
tokenizer = tiktoken.get_encoding(CONFIG['encoding'])

def count_tokens(text: str) -> int:
    """Count tokens using tiktoken"""
    return len(tokenizer.encode(text))

def create_chunk_id(doc_id: str, strategy: str, index: int) -> str:
    """Generate unique chunk ID"""
    return f"{doc_id}_{strategy}_{index:04d}"

def doc_to_metadata(doc: StripeDoc, exclude_content: bool = True) -> Dict:
    """Convert document to metadata dict"""
    metadata = asdict(doc)
    if exclude_content:
        metadata.pop('content', None)
        metadata.pop('code_blocks', None)
    return metadata

print("✓ Utility functions defined")

### Strategy 1: Fixed-Size Chunking with Overlap

In [ ]:
def chunk_fixed_size(docs: List[StripeDoc], 
                     chunk_size: int = 512, 
                     overlap: int = 50) -> List[Chunk]:
    """
    Fixed-size chunking with character overlap.
    Simple and predictable, good baseline.
    """
    splitter = CharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separator="\n\n",
        length_function=len
    )
    
    chunks = []
    strategy = "fixed_size"
    
    for doc in tqdm(docs, desc="Fixed-size chunking"):
        texts = splitter.split_text(doc.content)
        
        for idx, text in enumerate(texts):
            chunk = Chunk(
                chunk_id=create_chunk_id(doc.doc_id, strategy, idx),
                doc_id=doc.doc_id,
                content=text,
                chunk_index=idx,
                chunking_strategy=strategy,
                token_count=count_tokens(text),
                char_count=len(text),
                metadata=doc_to_metadata(doc)
            )
            chunks.append(chunk)
    
    return chunks

# Execute
fixed_chunks = chunk_fixed_size(stripe_docs, chunk_size=512, overlap=50)
print(f"✓ Created {len(fixed_chunks)} fixed-size chunks")
print(f"  Avg tokens per chunk: {np.mean([c.token_count for c in fixed_chunks]):.1f}")
print(f"  Avg chars per chunk: {np.mean([c.char_count for c in fixed_chunks]):.1f}")

### Strategy 2: Recursive Character Splitting

In [ ]:
def chunk_recursive(docs: List[StripeDoc], 
                   chunk_size: int = 512, 
                   overlap: int = 50) -> List[Chunk]:
    """
    Recursive character splitting - respects document structure.
    Tries to split on paragraphs, then sentences, then characters.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
        length_function=len
    )
    
    chunks = []
    strategy = "recursive"
    
    for doc in tqdm(docs, desc="Recursive chunking"):
        texts = splitter.split_text(doc.content)
        
        for idx, text in enumerate(texts):
            chunk = Chunk(
                chunk_id=create_chunk_id(doc.doc_id, strategy, idx),
                doc_id=doc.doc_id,
                content=text,
                chunk_index=idx,
                chunking_strategy=strategy,
                token_count=count_tokens(text),
                char_count=len(text),
                metadata=doc_to_metadata(doc)
            )
            chunks.append(chunk)
    
    return chunks

# Execute
recursive_chunks = chunk_recursive(stripe_docs, chunk_size=512, overlap=50)
print(f"✓ Created {len(recursive_chunks)} recursive chunks")
print(f"  Avg tokens per chunk: {np.mean([c.token_count for c in recursive_chunks]):.1f}")
print(f"  Avg chars per chunk: {np.mean([c.char_count for c in recursive_chunks]):.1f}")

### Strategy 3: Semantic Chunking with LlamaIndex

In [ ]:
def chunk_semantic(docs: List[StripeDoc], 
                  buffer_size: int = 1,
                  breakpoint_percentile_threshold: int = 95) -> List[Chunk]:
    """
    Semantic chunking using embeddings to find natural breakpoints.
    Chunks are created where semantic similarity drops significantly.
    """
    # Initialize embedding model
    embed_model = OpenAIEmbedding(model="text-embedding-ada-002")
    
    # Create semantic splitter
    splitter = SemanticSplitterNodeParser(
        buffer_size=buffer_size,
        breakpoint_percentile_threshold=breakpoint_percentile_threshold,
        embed_model=embed_model
    )
    
    chunks = []
    strategy = "semantic"
    
    for doc in tqdm(docs, desc="Semantic chunking"):
        # Convert to LlamaIndex document
        llama_doc = LlamaDocument(
            text=doc.content,
            metadata=doc_to_metadata(doc)
        )
        
        # Split into nodes
        nodes = splitter.get_nodes_from_documents([llama_doc])
        
        for idx, node in enumerate(nodes):
            chunk = Chunk(
                chunk_id=create_chunk_id(doc.doc_id, strategy, idx),
                doc_id=doc.doc_id,
                content=node.text,
                chunk_index=idx,
                chunking_strategy=strategy,
                token_count=count_tokens(node.text),
                char_count=len(node.text),
                metadata=doc_to_metadata(doc)
            )
            chunks.append(chunk)
    
    return chunks

print("Note: Semantic chunking requires embedding generation and may take several minutes...")
semantic_chunks = chunk_semantic(stripe_docs, buffer_size=1, breakpoint_percentile_threshold=95)
print(f"✓ Created {len(semantic_chunks)} semantic chunks")
print(f"  Avg tokens per chunk: {np.mean([c.token_count for c in semantic_chunks]):.1f}")
print(f"  Avg chars per chunk: {np.mean([c.char_count for c in semantic_chunks]):.1f}")

### Strategy 4: Sentence-Window Retrieval

In [ ]:
def chunk_sentence_window(docs: List[StripeDoc],
                         window_size: int = 3,
                         sentence_splitter_chunk_size: int = 1024) -> List[Chunk]:
    """
    Sentence-window chunking using LlamaIndex's SentenceWindowNodeParser.
    Each chunk is a sentence, with surrounding sentences stored as metadata.
    Embeddings are generated for the core sentence only (focused semantic representation).
    """
    from llama_index.core.node_parser import SentenceWindowNodeParser
    from llama_index.core import Document as LlamaDocument

    # Create sentence window parser
    # This automatically handles sentence splitting and window metadata
    node_parser = SentenceWindowNodeParser.from_defaults(
        window_size=window_size,  # Number of sentences on each side
        window_metadata_key="window",  # Key for window context in metadata
        original_text_metadata_key="original_sentence",  # Key for core sentence
    )

    chunks = []
    strategy = "sentence_window"

    for doc in tqdm(docs, desc="Sentence-window chunking (LlamaIndex)"):
        # Convert to LlamaIndex document
        llama_doc = LlamaDocument(
            text=doc.content,
            metadata=doc_to_metadata(doc)
        )

        # Parse into sentence-window nodes
        nodes = node_parser.get_nodes_from_documents([llama_doc])

        for idx, node in enumerate(nodes):
            # The node.text is the core sentence (what gets embedded)
            # The node.metadata['window'] contains the surrounding context

            metadata = doc_to_metadata(doc)
            metadata['window_context'] = node.metadata.get('window', '')
            metadata['window_size'] = window_size
            metadata['sentence_index'] = idx
            metadata['original_sentence'] = node.metadata.get('original_sentence', node.text)

            # Create chunk
            chunk = Chunk(
                chunk_id=create_chunk_id(doc.doc_id, strategy, idx),
                doc_id=doc.doc_id,
                content=node.text,  # Core sentence only - this is what gets embedded
                chunk_index=idx,
                chunking_strategy=strategy,
                token_count=count_tokens(node.text),
                char_count=len(node.text),
                metadata=metadata
            )
            chunks.append(chunk)

    return chunks

# Execute
sentence_window_chunks = chunk_sentence_window(stripe_docs, window_size=3)
print(f"✓ Created {len(sentence_window_chunks)} sentence-window chunks (LlamaIndex)")
print(f"  Avg tokens per chunk: {np.mean([c.token_count for c in sentence_window_chunks]):.1f}")
print(f"  Avg chars per chunk: {np.mean([c.char_count for c in sentence_window_chunks]):.1f}")
print(f"\n  Note: Embeddings will be generated for core sentences only.")
print(f"        Window context is stored in metadata for LLM generation.")

### Strategy 5: Hierarchical Parent-Child Chunking

In [ ]:
def chunk_hierarchical(docs: List[StripeDoc],
                      chunk_sizes: List[int] = [2048, 512, 128]) -> List[Chunk]:
    """
    Hierarchical chunking: Creates parent-child relationships.
    Allows retrieval at different granularities.
    """
    # Create node parsers for each level
    node_parser = HierarchicalNodeParser.from_defaults(
        chunk_sizes=chunk_sizes
    )
    
    all_chunks = []
    strategy = "hierarchical"

    level_name = ['parent', 'child', 'grandchild']

    for doc in tqdm(docs, desc="Hierarchical chunking"):
        # Convert to LlamaIndex document
        llama_doc = LlamaDocument(
            text=doc.content,
            metadata=doc_to_metadata(doc)
        )
        
        # Get hierarchical nodes
        nodes = node_parser.get_nodes_from_documents([llama_doc])
        
        # Build chunk hierarchy
        chunk_map = {}

        for node in nodes:
            node_id = node.node_id
            
            # Determine level from metadata
            level = 0  # Default
            if hasattr(node, 'metadata') and 'chunk_size' in node.metadata:
                chunk_size = node.metadata['chunk_size']
                level = chunk_sizes.index(chunk_size) if chunk_size in chunk_sizes else 0
            
            chunk_idx = len([c for c in chunk_map.values() if c.metadata.get('level') == level])
            
            metadata = doc_to_metadata(doc)
            metadata['level'] = level
            metadata['hierarchy_level_name'] = level_name[min(level, 2)]
            
            chunk = Chunk(
                chunk_id=create_chunk_id(doc.doc_id, f"{strategy}_L{level}", chunk_idx),
                doc_id=doc.doc_id,
                content=node.text,
                chunk_index=chunk_idx,
                chunking_strategy=strategy,
                token_count=count_tokens(node.text),
                char_count=len(node.text),
                metadata=metadata
            )
            
            # Store parent-child relationships
            if hasattr(node, 'parent_node') and node.parent_node:
                chunk.parent_chunk_id = node.parent_node.node_id
            
            if hasattr(node, 'child_nodes') and node.child_nodes:
                chunk.child_chunk_ids = [child.node_id for child in node.child_nodes]
            
            chunk_map[node_id] = chunk
        
        all_chunks.extend(chunk_map.values())
    
    return all_chunks

# Execute
hierarchical_chunks = chunk_hierarchical(stripe_docs, chunk_sizes=[2048, 512, 128])
print(f"✓ Created {len(hierarchical_chunks)} hierarchical chunks")
print(f"  Avg tokens per chunk: {np.mean([c.token_count for c in hierarchical_chunks]):.1f}")
print(f"  Avg chars per chunk: {np.mean([c.char_count for c in hierarchical_chunks]):.1f}")

# Show distribution by level
level_counts = {}
for chunk in hierarchical_chunks:
    level = chunk.metadata.get('level', 'unknown')
    level_counts[level] = level_counts.get(level, 0) + 1
print(f"\n  Distribution by hierarchy level:")
for level, count in sorted(level_counts.items()):
    level_name = ['parent', 'child', 'grandchild'][min(level, 2)] if isinstance(level, int) else level
    print(f"    Level {level} ({level_name}): {count} chunks")

## Chunking Strategy Comparison

In [ ]:
# Collect all chunking results
all_chunking_results = {
    'fixed_size': fixed_chunks,
    'recursive': recursive_chunks,
    'semantic': semantic_chunks,
    'sentence_window': sentence_window_chunks,
    'hierarchical': hierarchical_chunks
}

# Comparative analysis
comparison_data = []

for strategy, chunks in all_chunking_results.items():
    token_counts = [c.token_count for c in chunks]
    char_counts = [c.char_count for c in chunks]
    
    comparison_data.append({
        'strategy': strategy,
        'total_chunks': len(chunks),
        'avg_tokens': np.mean(token_counts),
        'std_tokens': np.std(token_counts),
        'min_tokens': np.min(token_counts),
        'max_tokens': np.max(token_counts),
        'avg_chars': np.mean(char_counts),
        'chunks_per_doc': len(chunks) / len(stripe_docs)
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.round(2)

print("\nChunking Strategy Comparison")
print("=" * 100)
print(comparison_df.to_string(index=False))

# Save comparison
comparison_df.to_csv(CONFIG['output_dir'] / 'chunking_comparison.csv', index=False)
print(f"\n✓ Comparison saved to {CONFIG['output_dir'] / 'chunking_comparison.csv'}")

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Total chunks
comparison_df.plot(x='strategy', y='total_chunks', kind='bar', ax=axes[0, 0], 
                   color='steelblue', legend=False)
axes[0, 0].set_title('Total Chunks per Strategy', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Strategy')
axes[0, 0].set_ylabel('Number of Chunks')
axes[0, 0].tick_params(axis='x', rotation=45)

# Average token count
comparison_df.plot(x='strategy', y='avg_tokens', kind='bar', ax=axes[0, 1], 
                   color='coral', legend=False)
axes[0, 1].set_title('Average Tokens per Chunk', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Strategy')
axes[0, 1].set_ylabel('Tokens')
axes[0, 1].tick_params(axis='x', rotation=45)

# Token distribution (box plot)
token_data = [all_chunking_results[strategy] for strategy in comparison_df['strategy']]
token_counts_by_strategy = [[c.token_count for c in chunks] for chunks in token_data]
axes[1, 0].boxplot(token_counts_by_strategy, tick_labels=comparison_df['strategy'])
axes[1, 0].set_title('Token Distribution by Strategy', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Strategy')
axes[1, 0].set_ylabel('Tokens')
axes[1, 0].tick_params(axis='x', rotation=45)

# Chunks per document
comparison_df.plot(x='strategy', y='chunks_per_doc', kind='bar', ax=axes[1, 1], 
                   color='lightgreen', legend=False)
axes[1, 1].set_title('Average Chunks per Document', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Strategy')
axes[1, 1].set_ylabel('Chunks per Document')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(CONFIG['output_dir'] / 'chunking_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Comparison visualizations saved")

### Export for Vector Databases

Prepare chunks in formats optimized for ChromaDB, Pinecone, and Weaviate.

In [ ]:
def export_for_chromadb(chunks: List[Chunk], strategy_name: str):
    """Export chunks in ChromaDB-friendly format"""
    export_data = {
        'documents': [c.content for c in chunks],
        'metadatas': [c.metadata for c in chunks],
        'ids': [c.chunk_id for c in chunks]
    }
    
    output_file = CONFIG['output_dir'] / CONFIG['chunks_dir'] / f'chromadb_{strategy_name}.json'
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)
    
    return output_file

def export_for_pinecone(chunks: List[Chunk], strategy_name: str):
    """Export chunks in Pinecone-friendly format"""
    export_data = []
    
    for chunk in chunks:
        vector_data = {
            'id': chunk.chunk_id,
            'values': [],  # Embeddings will be added during ingestion
            'metadata': {
                **chunk.metadata,
                'text': chunk.content,  # Pinecone stores text in metadata
                'chunk_index': chunk.chunk_index,
                'token_count': chunk.token_count,
                'chunking_strategy': chunk.chunking_strategy
            }
        }
        export_data.append(vector_data)
    
    output_file = CONFIG['output_dir'] / CONFIG['chunks_dir'] / f'pinecone_{strategy_name}.json'
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)
    
    return output_file

def export_for_weaviate(chunks: List[Chunk], strategy_name: str):
    """Export chunks in Weaviate-friendly format (GraphQL schema)"""
    export_data = []
    
    for chunk in chunks:
        object_data = {
            'class': 'StripeDocChunk',
            'id': chunk.chunk_id,
            'properties': {
                'content': chunk.content,
                'doc_id': chunk.doc_id,
                'chunk_index': chunk.chunk_index,
                'chunking_strategy': chunk.chunking_strategy,
                'token_count': chunk.token_count,
                'char_count': chunk.char_count,
                'doc_type': chunk.metadata.get('doc_type'),
                'category': chunk.metadata.get('category'),
                'subcategory': chunk.metadata.get('subcategory'),
                'url': chunk.metadata.get('url'),
                'title': chunk.metadata.get('title')
            }
        }
        export_data.append(object_data)
    
    output_file = CONFIG['output_dir'] / CONFIG['chunks_dir'] / f'weaviate_{strategy_name}.json'
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)
    
    return output_file

# Export all strategies
print("Exporting chunks for vector databases...\n")

for strategy_name, chunks in all_chunking_results.items():
    print(f"Exporting {strategy_name}...")
    
    chromadb_file = export_for_chromadb(chunks, strategy_name)
    print(f"  ✓ ChromaDB: {chromadb_file}")
    
    pinecone_file = export_for_pinecone(chunks, strategy_name)
    print(f"  ✓ Pinecone: {pinecone_file}")
    
    weaviate_file = export_for_weaviate(chunks, strategy_name)
    print(f"  ✓ Weaviate: {weaviate_file}")
    
    print()

print("✓ All exports complete!")

### Sample Chunks for Quality Check

In [ ]:
# Display sample chunks from each strategy
def display_sample_chunk(chunks: List[Chunk], strategy_name: str, sample_idx: int = 0):
    """Display a sample chunk for manual quality inspection"""
    if sample_idx >= len(chunks):
        sample_idx = 0
    
    chunk = chunks[sample_idx]
    
    print(f"\n{'='*80}")
    print(f"Strategy: {strategy_name.upper()}")
    print(f"{'='*80}")
    print(f"Chunk ID: {chunk.chunk_id}")
    print(f"Doc Type: {chunk.metadata.get('doc_type')}")
    print(f"Category: {chunk.metadata.get('category')}")
    print(f"Tokens: {chunk.token_count} | Chars: {chunk.char_count}")
    print(f"\nContent Preview:")
    print("-" * 80)
    # Show first 500 characters
    preview = chunk.content[:500]
    if len(chunk.content) > 500:
        preview += "..."
    print(preview)
    print("-" * 80)

# Display samples
print("\nSAMPLE CHUNKS FROM EACH STRATEGY")
print("=" * 80)

for strategy_name, chunks in all_chunking_results.items():
    # Try to find a chunk about payment intents for consistency
    sample_idx = 0
    for idx, chunk in enumerate(chunks):
        if 'payment' in chunk.content.lower() and len(chunk.content) > 200:
            sample_idx = idx
            break
    
    display_sample_chunk(chunks, strategy_name, sample_idx)